# Fig. 1 WT Fas3/DAPI StarDist segmentation

Cleaned from `RingCanals/Code/StardistPrediction_Fas3_DAPI_RGB.ipynb`. This notebook segments Fas3-positive follicle-cell boundaries in WT image stacks used for the Fig. 1 ring-canal-to-cell quantification workflow.

The raw image stacks are not stored in this publication repo. Set `data_directory` to the local folder containing ImageJ-exported `.tif` stacks before running. The source notebook loaded the `Fas3_round9` StarDist model; that model is archived locally in `../Models/Stardist/Fas3_round9`.


## Imports

Figure association: upstream segmentation for Fig. 1D, Fig. 1E, Fig. 1L, and Supp. Fig. 2E measurements.


In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import matplotlib
matplotlib.rcParams["image.interpolation"] = "None"
import matplotlib.pyplot as plt
import numpy as np
from csbdeep.io import save_tiff_imagej_compatible
from csbdeep.utils import normalize
from skimage import filters
from stardist import random_label_cmap
from stardist.models import StarDist3D
from tifffile import imread

from utils.MergeLabels import MergeLabels

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

np.random.seed(6)
lbl_cmap = random_label_cmap()


## Paths and channel settings

Update only this cell when running the pipeline on a new image folder.


In [ ]:
# Fig. 1 WT input images and segmentation outputs.
data_directory = Path("../Images/Shrb_labeling/Control/Replicate1/Results")
results_directory = data_directory / "RGB_model"
expt_name = "Shrb_Control_Replicate2"

# Channel indices in the input TIFF stacks. The source files are ZYXC.
FAS3_CHANNEL = 0
DAPI_CHANNEL = 2

# Local copy of the StarDist model used by the original notebook.
model_basedir = Path("../Models/Stardist")
model_name = "Fas3_round9"

results_directory.mkdir(parents=True, exist_ok=True)
img_files = sorted(data_directory.glob("*.tif"))
img_list = [imread(path) for path in img_files]

print(f"Loaded {len(img_list)} image stacks from {data_directory}")


## Load StarDist model


In [ ]:
# Fig. 1 segmentation model. StarDist expects the model folder under model_basedir/model_name.
model = StarDist3D(None, name=model_name, basedir=str(model_basedir))


## Predict follicle-cell labels


In [ ]:
# Fig. 1 segmentation: build the RGB-style Fas3/blank/DAPI input used by the trained model.
axis_norm = (0, 1, 2)
labels_array = []
details_array = []

for img_raw in img_list:
    gc.collect(2)
    model_input = np.stack(
        [
            img_raw[..., FAS3_CHANNEL],
            np.zeros_like(img_raw[..., FAS3_CHANNEL]),
            img_raw[..., DAPI_CHANNEL],
        ],
        axis=-1,
    )
    model_input = normalize(model_input, 1, 99.8, axis=axis_norm)
    labels, details = model.predict_instances(model_input)
    labels_array.append(labels)
    details_array.append(details)

print(f"Predicted labels for {len(labels_array)} image stacks")


## Merge over-split labels


In [ ]:
# Fig. 1 cleanup: merge labels whose shrunken 3D bounding boxes still overlap.
labels_array = [MergeLabels(labels, shrink_factors=(0.5, 0.5)) for labels in labels_array]


## Quality-control preview


In [ ]:
# Fig. 1 QC: inspect center z-slices before saving ImageJ-compatible stacks.
n_preview = min(3, len(labels_array))
fig, axes = plt.subplots(n_preview, 2, figsize=(10, 3.5 * n_preview), squeeze=False)

for row in range(n_preview):
    z = labels_array[row].shape[0] // 2
    axes[row, 0].imshow(labels_array[row][z], cmap=lbl_cmap)
    axes[row, 1].imshow(img_list[row][z, :, :, FAS3_CHANNEL], cmap="gray")
    axes[row, 0].set_title("Predicted labels")
    axes[row, 1].set_title("Fas3 channel")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")

plt.tight_layout()


## Save combined image + label stacks


In [ ]:
# Fig. 1 output: append labels as a final channel for downstream ImageJ/Fiji review.
for i, labels in enumerate(labels_array, start=1):
    label_channel = labels[..., np.newaxis] if labels.ndim == 3 else labels
    combined = np.concatenate((img_list[i - 1], label_channel), axis=3)
    save_path = results_directory / f"{expt_name}_{i}.tif"
    save_tiff_imagej_compatible(save_path, combined, axes="ZYXC")
    print(f"Saved {save_path}")
